## ファイルの前処理

In [1]:
import os, requests
from Bio import PDB
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [2]:
base_dir = "/home/shaeo/cadd_training/20250927_GROMACS"

In [3]:
# ファイルのダウンロード（4ZJ8: OX1）
pdb_id = "3HTB"
pdb_id = pdb_id.lower()
url = f"https://files.rcsb.org/download/3HTB.pdb"
os.makedirs(base_dir + "/data", exist_ok=True)
out_file = base_dir + f"/data/{pdb_id}.pdb"

response = requests.get(url)
if response.status_code == 200:
    with open(out_file, "wb") as f:
        f.write(response.content)
else:
    print(f"Failed to fetch {pdb_id} from OPM (status {response.status_code})")

In [4]:
# リガンドとレセプターの分割
protein_file = base_dir + f"/data/receptor_{pdb_id}.pdb"
ligand_file = base_dir + f"/data/ligand_{pdb_id}.pdb"
ligand_resname = "JZ4"

parser = PDB.PDBParser(QUIET=True)
structure = parser.get_structure("complex", out_file)
io = PDB.PDBIO()

# レセプターを抽出
class ProteinSelect(PDB.Select):
    def accept_residue(self, residue):
        if not PDB.is_aa(residue, standard=True): # アミノ酸かどうか
            return False
        return True
io.set_structure(structure)
io.save(protein_file, ProteinSelect())

# リガンドを抽出
class LigandSelect(PDB.Select):
    def accept_residue(self, residue):
        return residue.get_resname().strip() == ligand_resname
io.set_structure(structure)
io.save(ligand_file, LigandSelect())

In [5]:
# リガンドのファイル形式を変換
ligand_sdf = base_dir + f"/data/ligand_{pdb_id}.sdf"
!obabel $ligand_file -O $ligand_sdf -p 7.4

1 molecule converted


In [6]:
# PDBfixer
protein_file_fixed = base_dir + f"/data/receptor_{pdb_id}-fixed.pdb"

fixer = PDBFixer(filename=protein_file)
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.4)

with open(protein_file_fixed, "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)

## 複合体の準備

In [7]:
import numpy as np
from rdkit import Chem
import mdtraj as md
import openmm as mm
import openmm.app as app
from openmm import unit
from openff.toolkit.topology import Molecule
from openmmforcefields.generators import GAFFTemplateGenerator
import parmed as pmd

In [8]:
gromacs_top = base_dir + "/data/system_gromacs.top"
gromacs_gro = base_dir + "/data/system_gromacs.gro"

In [9]:
# リガンドの前処理
ligname = "LIG"
supplier = Chem.SDMolSupplier(ligand_sdf, removeHs=False)
rdkit_mol = supplier[0]

# OpenFFのオブジェクトに変換
off_mol = Molecule.from_rdkit(rdkit_mol)    
off_mol.name = ligname  # リガンドの名前を指定

# 原子番号の指定
element_counter_dict = {}
for off_atom, rdkit_atom in zip(off_mol.atoms, rdkit_mol.GetAtoms()):
    element = rdkit_atom.GetSymbol()
    if element in element_counter_dict.keys():
        element_counter_dict[element] += 1
    else:
        element_counter_dict[element] = 1
    off_atom.name = element + str(element_counter_dict[element])

# OpenMMのオブジェクトに変換
off_mol_topology = off_mol.to_topology()
mol_topology = off_mol_topology.to_openmm()
mol_positions = off_mol.conformers[0]
mol_positions = mol_positions.to("nanometers")  # 単位を変換：Å → nm
omm_mol = app.Modeller(mol_topology, mol_positions)

In [10]:
# タンパク質の読み込み
fixer = PDBFixer(filename=protein_file_fixed)

# リガンド-タンパク質複合体の形成
md_protein_topology = md.Topology.from_openmm(fixer.topology)  # PDBfixerのオブジェクトを利用
md_ligand_topology = md.Topology.from_openmm(omm_mol.topology)
md_complex_topology = md_protein_topology.join(md_ligand_topology)
complex_topology = md_complex_topology.to_openmm()  # OpenMMのオブジェクトに変換

# 座標の更新
total_atoms = len(fixer.positions) + len(omm_mol.positions)
complex_positions = unit.Quantity(np.zeros([total_atoms, 3]), unit=unit.nanometers) # OpenMMで利用される配列
complex_positions[: len(fixer.positions)] = fixer.positions # タンパク質の座標を追加
complex_positions[len(fixer.positions) :] = omm_mol.positions   # リガンドの座標を追加

/home/shaeo/miniconda3/envs/openmm/lib/python3.10/site-packages/openmm/unit/quantity.py:752: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  self._value[key] = value / self.unit


In [11]:
# 力場の設定
protein_ff = "amber14-all.xml"    # タンパク質の力場（膜脂質の力場を含む）
solvent_ff = "amber14/tip3pfb.xml"    # 水の力場
forcefield = app.ForceField(protein_ff, solvent_ff) 
sage = GAFFTemplateGenerator(
    molecules=Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True), # リガンドの力場/電荷を設定
)
forcefield.registerTemplateGenerator(sage.generator)

In [12]:
# 複合体に水/イオンを追加
modeller = app.Modeller(complex_topology, complex_positions)
modeller.addSolvent(
    forcefield,
    boxSize=mm.Vec3(8, 8, 8)*unit.nanometers,
    positiveIon="Na+",
    negativeIon="Cl-",              
    ionicStrength=0.15*unit.molar
)

In [13]:
# PDBファイルとして保存
topology = modeller.topology
positions = modeller.positions
complex_file = base_dir + f"/data/system_{pdb_id}.pdb"
with open(complex_file, "w") as f:
    PDBFile.writeFile(topology, positions, f)

In [14]:
# GROMACS用のファイルへ変換
export_system = forcefield.createSystem(
    modeller.topology,
    # constraints=None,
    # rigidWater=False,
    flexibleConstraints=True
)
pmd_complex_struct = pmd.openmm.load_topology(topology, export_system, positions)
pmd_complex_struct.save(gromacs_top, overwrite=True)
pmd_complex_struct.save(gromacs_gro, overwrite=True)

## MDシミュレーション

In [15]:
# 環境変数の設定：自身の環境に合わせて変更
home_dir = os.environ["HOME"]
gmx_path = f"{home_dir}/opt/gromacs-2024.6/app/bin"
gmx_lib = f"{home_dir}/opt/gromacs-2024.6/app/lib"

os.environ["PATH"] = gmx_path + ":" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = gmx_lib + ":" + os.environ.get("LD_LIBRARY_PATH","")

In [16]:
result_dir = base_dir + "/result"
os.makedirs(result_dir, exist_ok=True)

### 1.エネルギー最小化

In [17]:
em_dir = result_dir + "/1_energy_minimization"
os.makedirs(em_dir, exist_ok=True)
em_result_prefix = em_dir + "/em"
em_mdp_path = em_result_prefix + ".mdp"
em_tpr_path = em_result_prefix + ".tpr"

In [18]:
# mdpファイルの作成
contents_mdp = """
; minim.mdp - used as input into grompp to generate em.tpr
; Parameters describing what to do, when to stop and what to save
integrator  = steep         ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0        ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.01          ; Minimization step size
nsteps      = 50000         ; Maximum number of (minimization) steps to perform

; Parameters describing how to find the neighbors of each atom and how to calculate the interactions
nstlist         = 20         ; Frequency to update the neighbor list and long range forces
cutoff-scheme   = Verlet    ; Buffered neighbor searching
; ns_type         = grid      ; Method to determine neighbor list (simple, grid)
coulombtype     = PME       ; Treatment of long range electrostatic interactions
rcoulomb        = 1.0       ; Short-range electrostatic cut-off
rvdw            = 1.0       ; Short-range Van der Waals cut-off
pbc             = xyz       ; Periodic Boundary Conditions in all 3 dimensions
"""
with open(em_mdp_path, mode="w") as f:
    f.write(contents_mdp)

In [19]:
# tprファイルの作成
!gmx grompp \
    -f $em_mdp_path \
    -c $gromacs_gro \
    -p $gromacs_top \
    -o $em_tpr_path

                      :-) GROMACS - gmx grompp, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx grompp -f /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em.mdp -c /home/shaeo/cadd_training/20250927_GROMACS/data/system_gromacs.gro -p /home/shaeo/cadd_training/20250927_GROMACS/data/system_gromacs.top -o /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em.tpr

Setting the LD random seed to -37879829

Generated 276 of the 276 non-bonded parameter combinations

Excluding 3 bonded neighbours molecule type 'system1'

Excluding 3 bonded neighbours molecule type 'UNK'

Excluding 3 bonded neighbours molecule type 'HOH'

Excluding 3 bonded neighbours molecule type 'NA'

Excluding 3 bonded neighbours molecule type 'CL'


Analysing residue names:
There are:   163    Protein residues
There are:     1      Other residues
There are: 15474      Water residues
There are:    90        Ion residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Number of degrees of freedom in T-Coupling group rest is 101019.00
The integrator does not provide a ensemble temperature, there is no system ensemble temperature

The largest distance between excluded atoms is 0.419 nm between atom 1876 and 1883
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 72x72x72, spacing 0.120 0.120 0.120

Estimate for the relative computational load of the PME mesh part: 0.29

This run will generate roughly 4 Mb of data

GROMACS reminds you: "Nobody Never Learnt No-Nothing from No History" (Gogol Bordello)



In [20]:
# 実行
!gmx mdrun -deffnm $em_result_prefix

                      :-) GROMACS - gmx mdrun, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx mdrun -deffnm /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em

Reading file /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em.tpr, VERSION 2024.6 (single precision)
1 GPU selected for this run.
Mapping of GPU IDs to the 1 GPU task in the 1 rank on this node:
  PP:0
PP tasks will do (non-perturbed) short-ranged interactions on the GPU
PP task will update and constrain coordinates on the CPU
Using 1 MPI thread
Using 16 OpenMP threads 


Steepest Descents:
   Tolerance (Fmax)   =  1.00000e+03
   Number of steps    =        50000

writing lowest energy coordinates.

Steepest Descents converged to Fmax < 1000 in 1695 steps
Potential Energy  = -8.8806144e+05
Maximum force     =  9.7970935e+02 on atom 26

In [21]:
# ポテンシャルエネルギーの変化
!echo 10 0 | gmx energy -f {em_result_prefix}.edr -o {em_result_prefix}.xvg

                      :-) GROMACS - gmx energy, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx energy -f /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em.edr -o /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em.xvg

Opened /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond             2  Angle            3  Proper-Dih.      4  Per.-Imp.-Dih.
  5  LJ-14            6  Coulomb-14       7  LJ-(SR)          8  Coulomb-(SR)  
  9  Coul.-recip.    10  Potential       11  Pressure        12  Vir

### 2.等温緩和

In [22]:
qd_dir = result_dir + "/2_quenched_dynamics"
os.makedirs(qd_dir, exist_ok=True)
qd_result_prefix = qd_dir + "/qd"
qd_mdp_path = qd_result_prefix + ".mdp"
qd_tpr_path = qd_result_prefix + ".tpr"

In [23]:
# mdpファイルの作成
contents_mdp = """
; --- 10 K, 10 steps (20 fs) ---
integrator      = md
dt              = 0.002
nsteps          = 10

; 速度生成（em.gro には速度が無い想定）
gen_vel         = yes
gen_temp        = 10
gen_seed        = -1

; 温度制御：全系まとめて簡単に
tcoupl          = V-rescale
tc-grps         = System
tau-t           = 0.1
ref-t           = 10

; 圧力制御なし（NVT）
pcoupl          = no

; 一般設定
constraints     = h-bonds
constraint-algorithm = lincs
cutoff-scheme   = Verlet
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

; 出力（軽量）
nstxout-compressed = 10
nstenergy      = 10
nstlog         = 10
"""
with open(qd_mdp_path, mode="w") as f:
    f.write(contents_mdp)

In [24]:
# tprファイルの作成
!gmx grompp \
    -f $qd_mdp_path \
    -c {em_result_prefix}.gro \
    -p $gromacs_top \
    -o $qd_tpr_path

                      :-) GROMACS - gmx grompp, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx grompp -f /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.mdp -c /home/shaeo/cadd_training/20250927_GROMACS/result/1_energy_minimization/em.gro -p /home/shaeo/cadd_training/20250927_GROMACS/data/system_gromacs.top -o /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.tpr


NOTE 1 [file /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.mdp]:
  Setting nstcalcenergy (100) equal to nstenergy (10)

Setting the LD random seed to -268437285

Generated 276 of the 276 non-bonded parameter combinations

Excluding 3 bonded neighbours molecule type 'system1'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'UNK'

turning H bonds into constraints...

Excludi


Calculated rlist for 1x1 atom pair-list as 1.001 nm, buffer size 0.001 nm

Set rlist, assuming 4x4 atom pair-list, to 1.000 nm, buffer size 0.000 nm

Note that mdrun will redetermine rlist based on the actual pair-list setup
Calculating fourier grid dimensions for X Y Z
Using a fourier grid of 72x72x72, spacing 0.120 0.120 0.120

Estimate for the relative computational load of the PME mesh part: 0.31

This run will generate roughly 4 Mb of data

There was 1 NOTE

GROMACS reminds you: "Philosophy of science is about as useful to scientists as ornithology is to birds." (Richard Feynman)



In [25]:
# 実行
!gmx mdrun -deffnm $qd_result_prefix

                      :-) GROMACS - gmx mdrun, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx mdrun -deffnm /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd

Reading file /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.tpr, VERSION 2024.6 (single precision)
Changing nstlist from 10 to 100, rlist from 1 to 1.01

Update groups can not be used for this system because atoms that are (in)directly constrained together are interdispersed with other atoms

1 GPU selected for this run.
Mapping of GPU IDs to the 2 GPU tasks in the 1 rank on this node:
  PP:0,PME:0
PP tasks will do (non-perturbed) short-ranged interactions on the GPU
PP task will update and constrain coordinates on the GPU
PME tasks will do all aspects on the GPU
Using 1 MPI thread
Using 8 OpenMP threads 

starting mdrun 'Generic title'

In [26]:
# ポテンシャルエネルギーの変化
!echo 10 0 | gmx energy -f {qd_result_prefix}.edr -o {qd_result_prefix}.xvg

                      :-) GROMACS - gmx energy, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx energy -f /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.edr -o /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.xvg

Opened /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond             2  Angle            3  Proper-Dih.      4  Per.-Imp.-Dih.
  5  LJ-14            6  Coulomb-14       7  LJ-(SR)          8  Coulomb-(SR)  
  9  Coul.-recip.    10  Potential       11  Kinetic-En.     12  Total-Ene

### 3.昇温

In [27]:
ht_dir = result_dir + "/3_heating"
os.makedirs(ht_dir, exist_ok=True)
ht_result_prefix = ht_dir + "/ht"
ht_mdp_path = ht_result_prefix + ".mdp"
ht_tpr_path = ht_result_prefix + ".tpr"

In [28]:
# mdpファイルの作成
contents_mdp = """
; --- NVT heating: 10 K -> 300 K over 500 ps ---
integrator              = md
dt                      = 0.002
nsteps                  = 500000           ; 1.0 ns

; 温度カップリング（3グループ：蛋白 / リガンド / 水+イオン）
tcoupl                  = V-rescale
tc-grps                 = Protein Other Water_and_ions
tau-t                   = 0.1     0.1   0.1
ref-t                   = 10      10    10

; 10K→300K アニーリング（各グループ同じカーブ）
annealing               = single  single single
annealing-npoints       = 2       2      2
annealing-time          = 0 500   0 500  0 500      ; ps
annealing-temp          = 10 300  10 300 10 300     ; K

; 圧力制御なし（NVT）
pcoupl                  = no

; 一般設定
constraints             = h-bonds
constraint-algorithm    = lincs
cutoff-scheme           = Verlet
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
pbc                     = xyz

; 出力（適宜調整）
nstxout-compressed      = 5000    ; 10 psごと
nstenergy               = 5000
nstlog                  = 5000
"""
with open(ht_mdp_path, mode="w") as f:
    f.write(contents_mdp)

In [29]:
# tprファイルの作成
!gmx grompp \
    -f $ht_mdp_path \
    -c {qd_result_prefix}.gro \
    -t {qd_result_prefix}.cpt \
    -p $gromacs_top \
    -o $ht_tpr_path

                      :-) GROMACS - gmx grompp, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx grompp -f /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.mdp -c /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.gro -t /home/shaeo/cadd_training/20250927_GROMACS/result/2_quenched_dynamics/qd.cpt -p /home/shaeo/cadd_training/20250927_GROMACS/data/system_gromacs.top -o /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.tpr

Setting the LD random seed to -1082167303

Generated 276 of the 276 non-bonded parameter combinations

Excluding 3 bonded neighbours molecule type 'system1'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'UNK'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'HOH'

turning H bonds into constraints...

Excludin

In [30]:
# 実行
!gmx mdrun -deffnm $ht_result_prefix

                      :-) GROMACS - gmx mdrun, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx mdrun -deffnm /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht

Reading file /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.tpr, VERSION 2024.6 (single precision)
Changing nstlist from 10 to 100, rlist from 1 to 1.01

Update groups can not be used for this system because atoms that are (in)directly constrained together are interdispersed with other atoms

1 GPU selected for this run.
Mapping of GPU IDs to the 2 GPU tasks in the 1 rank on this node:
  PP:0,PME:0
PP tasks will do (non-perturbed) short-ranged interactions on the GPU
PP task will update and constrain coordinates on the GPU
PME tasks will do all aspects on the GPU
Using 1 MPI thread
Using 8 OpenMP threads 

starting mdrun 'Generic title'
500000 steps,   100

In [31]:
# ポテンシャルエネルギーの変化
!echo 10 0 | gmx energy -f {ht_result_prefix}.edr -o {ht_result_prefix}.xvg

                      :-) GROMACS - gmx energy, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx energy -f /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.edr -o /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.xvg

Opened /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond             2  Angle            3  Proper-Dih.      4  Per.-Imp.-Dih.
  5  LJ-14            6  Coulomb-14       7  LJ-(SR)          8  Coulomb-(SR)  
  9  Coul.-recip.    10  Potential       11  Kinetic-En.     12  Total-Energy  
 13  Conserved-En.   14 

### 4.平衡化

In [32]:
eq_dir = result_dir + "/4_equibaration"
os.makedirs(eq_dir, exist_ok=True)
eq_result_prefix = eq_dir + "/eq"
eq_mdp_path = eq_result_prefix + ".mdp"
eq_tpr_path = eq_result_prefix + ".tpr"

In [33]:
# mdpファイルの作成
contents_mdp = """
; --- NPT 0.5 ns, no position restraints ---
integrator              = md
dt                      = 0.002
nsteps                  = 500000          ; 1.0 ns
; ※ 拘束マクロは入れない（define 行は不要）

; 速度：前段のチェックポイントから継続する前提
gen_vel                 = no
; （チェックポイントが無いなら yes / gen_temp=300 に変更）

; 温度制御（3グループ例：Protein / Other / Water_and_ions）
tcoupl                  = V-rescale
tc-grps                 = Protein Other Water_and_ions
tau-t                   = 0.1     0.1   0.1
ref-t                   = 300     300   300

; 圧力制御（等方）
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau-p                   = 5.0
ref-p                   = 1.0
compressibility         = 4.5e-5          ; TIP3P水の標準値

; 一般設定
constraints             = h-bonds
constraint-algorithm    = lincs
cutoff-scheme           = Verlet
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
pbc                     = xyz

; 出力
nstxout-compressed      = 5000            ; 10 psごと
nstenergy               = 5000
nstlog                  = 5000

"""
with open(eq_mdp_path, mode="w") as f:
    f.write(contents_mdp)

In [34]:
# tprファイルの作成
!gmx grompp \
    -f $eq_mdp_path \
    -c {ht_result_prefix}.gro \
    -t {ht_result_prefix}.cpt \
    -p $gromacs_top \
    -o $eq_tpr_path

                      :-) GROMACS - gmx grompp, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx grompp -f /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.mdp -c /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.gro -t /home/shaeo/cadd_training/20250927_GROMACS/result/3_heating/ht.cpt -p /home/shaeo/cadd_training/20250927_GROMACS/data/system_gromacs.top -o /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.tpr

Setting the LD random seed to -12713989

Generated 276 of the 276 non-bonded parameter combinations

Excluding 3 bonded neighbours molecule type 'system1'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'UNK'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'HOH'

turning H bonds into constraints...

Excluding 3 bonded n

Analysing residue names:
There are:   163    Protein residues
There are:     1      Other residues
There are: 15474      Water residues
There are:    90        Ion residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Number of degrees of freedom in T-Coupling group Protein is 6528.80
Number of degrees of freedom in T-Coupling group Other is 54.00
Number of degrees of freedom in T-Coupling group Water_and_ions is 93111.20

The largest distance between excluded atoms is 0.422 nm between atom 94 and 101

Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 300 K

Calculated rlist for 1x1 atom pair-list as 1.035 nm, buffer size 0.035 nm

Set rlist, assuming 4x4 atom pair-list, to 1.000 nm, buffer size 0.000 nm

Note that mdrun will redetermine rlist based on the actual pair-list setup

Reading Coordinates, Velocities and Box size from old trajectory

Will read whole trajectory
Last frame         -1 time 1000.000   

In [35]:
# 実行
!gmx mdrun -deffnm $eq_result_prefix

                      :-) GROMACS - gmx mdrun, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx mdrun -deffnm /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq

Reading file /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.tpr, VERSION 2024.6 (single precision)
Changing nstlist from 10 to 100, rlist from 1 to 1.169

Update groups can not be used for this system because atoms that are (in)directly constrained together are interdispersed with other atoms

1 GPU selected for this run.
Mapping of GPU IDs to the 2 GPU tasks in the 1 rank on this node:
  PP:0,PME:0
PP tasks will do (non-perturbed) short-ranged interactions on the GPU
PP task will update and constrain coordinates on the GPU
PME tasks will do all aspects on the GPU
Using 1 MPI thread
Using 8 OpenMP threads 

starting mdrun 'Generic title'
500000 s

In [36]:
# ポテンシャルエネルギーの変化
!echo 10 0 | gmx energy -f {eq_result_prefix}.edr -o {eq_result_prefix}.xvg

                      :-) GROMACS - gmx energy, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx energy -f /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.edr -o /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.xvg

Opened /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond             2  Angle            3  Proper-Dih.      4  Per.-Imp.-Dih.
  5  LJ-14            6  Coulomb-14       7  LJ-(SR)          8  Coulomb-(SR)  
  9  Coul.-recip.    10  Potential       11  Kinetic-En.     12  Total-Energy  
 13  Cons

### 5.生産

In [37]:
md_dir = result_dir + "/5_production"
os.makedirs(md_dir, exist_ok=True)
md_result_prefix = md_dir + "/md"
md_mdp_path = md_result_prefix + ".mdp"
md_tpr_path = md_result_prefix + ".tpr"

In [38]:
# mdpファイルの作成
contents_mdp = """
; --- NPT 0.5 ns, no position restraints ---
integrator              = md
dt                      = 0.002
nsteps                  = 5000000          ; 10.0 ns
; ※ 拘束マクロは入れない（define 行は不要）

; 速度：前段のチェックポイントから継続する前提
gen_vel                 = no
; （チェックポイントが無いなら yes / gen_temp=300 に変更）

; 温度制御（3グループ例：Protein / Other / Water_and_ions）
tcoupl                  = V-rescale
tc-grps                 = Protein Other Water_and_ions
tau-t                   = 0.1     0.1   0.1
ref-t                   = 300     300   300

; 圧力制御（等方）
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau-p                   = 5.0
ref-p                   = 1.0
compressibility         = 4.5e-5          ; TIP3P水の標準値

; 一般設定
constraints             = h-bonds
constraint-algorithm    = lincs
cutoff-scheme           = Verlet
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
pbc                     = xyz

; 出力
nstxout-compressed      = 5000            ; 10 psごと
nstenergy               = 5000
nstlog                  = 5000

"""
with open(md_mdp_path, mode="w") as f:
    f.write(contents_mdp)

In [39]:
# tprファイルの作成
!gmx grompp \
    -f $md_mdp_path \
    -c {eq_result_prefix}.gro \
    -t {eq_result_prefix}.cpt \
    -p $gromacs_top \
    -o $md_tpr_path

                      :-) GROMACS - gmx grompp, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx grompp -f /home/shaeo/cadd_training/20250927_GROMACS/result/5_production/md.mdp -c /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.gro -t /home/shaeo/cadd_training/20250927_GROMACS/result/4_equibaration/eq.cpt -p /home/shaeo/cadd_training/20250927_GROMACS/data/system_gromacs.top -o /home/shaeo/cadd_training/20250927_GROMACS/result/5_production/md.tpr

Setting the LD random seed to -537010945

Generated 276 of the 276 non-bonded parameter combinations

Excluding 3 bonded neighbours molecule type 'system1'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'UNK'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'HOH'

turning H bonds into constraints...

Excluding 3 b

Analysing residue names:
There are:   163    Protein residues
There are:     1      Other residues
There are: 15474      Water residues
There are:    90        Ion residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...
Number of degrees of freedom in T-Coupling group Protein is 6528.80
Number of degrees of freedom in T-Coupling group Other is 54.00
Number of degrees of freedom in T-Coupling group Water_and_ions is 93111.20

The largest distance between excluded atoms is 0.432 nm between atom 90 and 100

Determining Verlet buffer for a tolerance of 0.005 kJ/mol/ps at 300 K

Calculated rlist for 1x1 atom pair-list as 1.035 nm, buffer size 0.035 nm

Set rlist, assuming 4x4 atom pair-list, to 1.000 nm, buffer size 0.000 nm

Note that mdrun will redetermine rlist based on the actual pair-list setup

Reading Coordinates, Velocities and Box size from old trajectory

Will read whole trajectory
Last frame         -1 time 1000.000   

In [40]:
# 実行
!gmx mdrun -deffnm $md_result_prefix

                      :-) GROMACS - gmx mdrun, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx mdrun -deffnm /home/shaeo/cadd_training/20250927_GROMACS/result/5_production/md

Reading file /home/shaeo/cadd_training/20250927_GROMACS/result/5_production/md.tpr, VERSION 2024.6 (single precision)
Changing nstlist from 10 to 100, rlist from 1 to 1.171

Update groups can not be used for this system because atoms that are (in)directly constrained together are interdispersed with other atoms

1 GPU selected for this run.
Mapping of GPU IDs to the 2 GPU tasks in the 1 rank on this node:
  PP:0,PME:0
PP tasks will do (non-perturbed) short-ranged interactions on the GPU
PP task will update and constrain coordinates on the GPU
PME tasks will do all aspects on the GPU
Using 1 MPI thread
Using 8 OpenMP threads 

starting mdrun 'Generic title'
5000000 step

In [41]:
# ポテンシャルエネルギーの変化
!echo 10 0 | gmx energy -f {md_result_prefix}.edr -o {md_result_prefix}.xvg

                      :-) GROMACS - gmx energy, 2024.6 (-:

Executable:   /home/shaeo/opt/gromacs-2024.6/app/bin/gmx
Data prefix:  /home/shaeo/opt/gromacs-2024.6/app
Working dir:  /home/shaeo/cadd_training/20250927_GROMACS
Command line:
  gmx energy -f /home/shaeo/cadd_training/20250927_GROMACS/result/5_production/md.edr -o /home/shaeo/cadd_training/20250927_GROMACS/result/5_production/md.xvg

Opened /home/shaeo/cadd_training/20250927_GROMACS/result/5_production/md.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------------------------------------------------------------------
  1  Bond             2  Angle            3  Proper-Dih.      4  Per.-Imp.-Dih.
  5  LJ-14            6  Coulomb-14       7  LJ-(SR)          8  Coulomb-(SR)  
  9  Coul.-recip.    10  Potential       11  Kinetic-En.     12  Total-Energy  
 13  Conserved-

### 6.トラジェクトリの後処理

In [42]:
xtc_path_ht = ht_result_prefix + ".xtc"
xtc_path_eq = eq_result_prefix + ".xtc"
xtc_path_md = md_result_prefix + ".xtc"
fixed_path_ht = ht_result_prefix + "_pc.xtc"
fixed_path_eq = eq_result_prefix + "_pc.xtc"
fixed_path_md = md_result_prefix + "_pc.xtc"

In [43]:
dict_path_xtc = {
    xtc_path_ht:fixed_path_ht,
    xtc_path_eq:fixed_path_eq,
    xtc_path_md:fixed_path_md,
}

In [44]:
for xtc_path, fixed_xtc_path in dict_path_xtc.items():
    traj = md.load(xtc_path, top=gromacs_gro)
    traj_whole = traj.make_molecules_whole()    # 分子を “ちぎれない” 形に直す（make whole）
    prot = traj_whole.topology.select("protein")    # タンパクを基準にセンタリング＆ボックス内にラップ
    traj_centered = traj_whole.center_coordinates()
    traj_wrapped = traj_centered.image_molecules()  # 近接イメージに押し戻す
    traj_fit = traj_wrapped.superpose(traj_wrapped, frame=0, atom_indices=prot)     # 剛体合わせ（平行移動・回転除去）してRMSD等が滑らかに
    traj_fit.save_xtc(fixed_xtc_path)

## 解析

### MM/GBSA、MM/PBSA

In [57]:
from openmm.app.pdbfile import PDBFile
import parmed as pmd
from parmed.tools.actions import changeRadii
import pandas as pd

In [65]:
complex_prmtop_path = base_dir + "/data/complex.prmtop"
complex_inpcrd_path = base_dir + "/data/complex.inpcrd"
receptor_prmtop_path = base_dir + "/data/receptor.prmtop"
receptor_inpcrd_path = base_dir + "/data/receptor.inpcrd"
ligand_prmtop_path = base_dir + "/data/ligand.prmtop"
ligand_inpcrd_path = base_dir + "/data/ligand.inpcrd"
xtc_mm_path = base_dir + "/result/md_traj_for_mm.xtc"

In [66]:
# PBCを取得
with open(complex_file, 'r') as f:
    box = {}
    while True:
        line = f.readline()
        if not line: break
        
        contents = line.split()
        if contents[0] == "CRYST1":
            size = line.split()[1].strip('x="')
            box['A'] = float(contents[1])
            box['B'] = float(contents[2])
            box['C'] = float(contents[3])
        else:
            pass
vectors = [
    mm.Vec3(box["A"], 0.0, 0.0),
    mm.Vec3(0.0, box["B"], 0.0),
    mm.Vec3(0.0, 0.0, box["C"]),
] * unit.nanometer

In [67]:
# sytemの作成
pdb = PDBFile(complex_file)
modeller = app.Modeller(pdb.topology, pdb.positions)
modeller.deleteWater()  # 水分子を削除
to_delete = [res for res in modeller.topology.residues()
             if res.name in ["NA", "CL", "POP"]]
modeller.delete(to_delete)  # 特定の残基を削除
modeller.topology.setPeriodicBoxVectors(vectors)
topology, positions = modeller.topology, modeller.positions # PBC設定

forcefield = app.ForceField(protein_ff, solvent_ff) 
gaff = GAFFTemplateGenerator(
    molecules=Molecule.from_rdkit(rdkit_mol, allow_undefined_stereo=True) # リガンドの力場/電荷を設定
)
forcefield.registerTemplateGenerator(gaff.generator)
export_system = forcefield.createSystem(
    topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0*unit.nanometer,
    constraints="HBonds"
)

In [68]:
# ParmEd 変換
parm = pmd.openmm.load_topology(topology, export_system, positions)
parm.box = None
changeRadii(parm, "mbondi2").execute()

# 複合体を保存
parm.save(complex_prmtop_path, overwrite=True)
parm.save(complex_inpcrd_path, overwrite=True)

# 受容体のみを選択
parm_receptor = parm["(:1-163)"]    # result/em_structure.pdb
parm_receptor.save(receptor_prmtop_path, overwrite=True)
parm_receptor.save(receptor_inpcrd_path, overwrite=True)

# リガンドのみを選択（例: 残基名 LIG）
parm_ligand = parm["(:UNK)"]
parm_ligand.save(ligand_prmtop_path, overwrite=True)
parm_ligand.save(ligand_inpcrd_path, overwrite=True)

In [69]:
traj = md.load_xtc(fixed_path_md, top=gromacs_gro)
# タンパク + リガンドのみ残す
prot_lig = traj.atom_slice(traj.topology.select("protein or resname UNK"))
prot_lig.save_xtc(xtc_mm_path)

#### MM/GBSA

In [70]:
mmgbsa_in_path = base_dir + "/result/mmgbsa.in"
mmgbsa_result_path = base_dir + "/result/mmgbsa.dat"
mmgbsa_csv_path = base_dir + "/result/mmgbsa.csv"

In [71]:
mmgbsa_in = '''
&general
   startframe=1, 
   endframe=1000000,
   interval=1,
   keep_files=0, 
   verbose=1,
/
&gb
   igb=5, 
   saltcon=0.150,
/
'''
with open(mmgbsa_in_path, "w") as f:
    f.write(mmgbsa_in)

In [72]:
!MMPBSA.py -O -i $mmgbsa_in_path -o $mmgbsa_result_path \
  -cp $complex_prmtop_path \
  -rp $receptor_prmtop_path \
  -lp $ligand_prmtop_path \
  -y $xtc_mm_path

Loading and checking parameter files for compatibility...
cpptraj found! Using /home/shaeo/miniconda3/envs/openmm/bin/cpptraj
mmpbsa_py_energy found! Using /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
Preparing trajectories for simulation...
1001 frames were processed by cpptraj for use in calculation.

Running calculations on normal system...

Beginning GB calculations with /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
  calculating complex contribution...
  calculating receptor contribution...
  calculating ligand contribution...

Timing:
Total setup time:                           0.003 min.
Creating trajectories with cpptraj:         0.053 min.
Total calculation time:                     8.489 min.

Total GB calculation time:                  8.489 min.

Statistics calculation & output writing:    0.000 min.
Total time taken:                           8.545 min.


MMPBSA.py Finished! Thank you for using. Please cite us if you publish this work with this paper:


In [73]:
list_data = []
list_col = []
with open(mmgbsa_result_path) as f:
    while True:
        line = f.readline()
        if "Differences (Complex - Receptor - Ligand):" in line:
            line = f.readline()
            line = f.readline()
            break
        if not line: 
            break
        
    flag = True
    while flag:
        line = f.readline()
        parts = line.split()
        if len(parts) >= 4:
            if len(parts) == 4:
                para_name = parts[0]
            elif len(parts) == 5:
                para_name = parts[0] + "_" + parts[1]
            elif len(parts) == 6:
                para_name = parts[0] + "_" + parts[1] + "_" + parts[2]
            else:
                raise("unexpected error")
            if para_name == "DELTA_TOTAL":
                flag = False
            list_data.append(float(parts[-3]))
            list_data.append(float(parts[-2]))
            list_data.append(float(parts[-1]))
            list_col.append("MMGBSA_"+para_name)
            list_col.append("MMGBSA_"+para_name+"_STDD")
            list_col.append("MMGBSA_"+para_name+"_STDE")
df_mmgbsa = pd.DataFrame(data=np.array(list_data).reshape(1, -1), columns=list_col, index=["ligand"])
df_mmgbsa.to_csv(mmgbsa_csv_path)

#### MM/PBSA

In [74]:
mmpbsa_in_path = base_dir + "/result/mmpbsa.in"
mmpbsa_result_path = base_dir + "/result/mmpbsa.dat"
mmpbsa_csv_path = base_dir + "/result/mmpbsa.csv"

In [78]:
mmpbsa_in = '''
&general
   startframe=1, 
   endframe=10000, 
   interval=5,
   keep_files=0, 
   verbose=1,
/
&pb
   istrng=0.150,
   indi=20.0, 
   exdi=80.0,
   inp=2, 
   radiopt=0,
   fillratio=4.0, 
   scale=2.0,
/
'''
with open(mmpbsa_in_path, "w") as f:
    f.write(mmpbsa_in)

In [79]:
!MMPBSA.py -O -i $mmpbsa_in_path -o $mmpbsa_result_path \
  -cp $complex_prmtop_path \
  -rp $receptor_prmtop_path \
  -lp $ligand_prmtop_path \
  -y $xtc_mm_path

Loading and checking parameter files for compatibility...
cpptraj found! Using /home/shaeo/miniconda3/envs/openmm/bin/cpptraj
mmpbsa_py_energy found! Using /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
Preparing trajectories for simulation...
201 frames were processed by cpptraj for use in calculation.

Running calculations on normal system...

Beginning PB calculations with /home/shaeo/miniconda3/envs/openmm/bin/mmpbsa_py_energy
  calculating complex contribution...
  calculating receptor contribution...
  calculating ligand contribution...

Timing:
Total setup time:                           0.003 min.
Creating trajectories with cpptraj:         0.012 min.
Total calculation time:                    25.367 min.

Total PB calculation time:                 25.367 min.

Statistics calculation & output writing:    0.000 min.
Total time taken:                          25.382 min.


MMPBSA.py Finished! Thank you for using. Please cite us if you publish this work with this paper:
 

In [77]:
list_data = []
list_col = []
with open(mmpbsa_result_path) as f:
    while True:
        line = f.readline()
        if "Differences (Complex - Receptor - Ligand):" in line:
            line = f.readline()
            line = f.readline()
            break
        if not line: 
            break
        
    flag = True
    while flag:
        line = f.readline()
        parts = line.split()
        if len(parts) >= 4:
            if len(parts) == 4:
                para_name = parts[0]
            elif len(parts) == 5:
                para_name = parts[0] + "_" + parts[1]
            elif len(parts) == 6:
                para_name = parts[0] + "_" + parts[1] + "_" + parts[2]
            else:
                raise("unexpected error")
            if para_name == "DELTA_TOTAL":
                flag = False
            list_data.append(float(parts[-3]))
            list_data.append(float(parts[-2]))
            list_data.append(float(parts[-1]))
            list_col.append("MMPBSA_"+para_name)
            list_col.append("MMPBSA_"+para_name+"_STDD")
            list_col.append("MMPBSA_"+para_name+"_STDE")
df_mmpbsa = pd.DataFrame(data=np.array(list_data).reshape(1, -1), columns=list_col, index=["ligand"])
df_mmpbsa.to_csv(mmpbsa_csv_path)